In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [7]:
import os
!pip install dagshub mlflow -q
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

import dagshub
dagshub.init(repo_owner='lkhiz23', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

import mlflow
print("Connected ✓")

Initialized MLflow to track repo "lkhiz23/IEEE-CIS-Fraud-Detection"

Repository lkhiz23/IEEE-CIS-Fraud-Detection initialized!

Connected ✓


# Model Experiment - Random Forest

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PATH = '/kaggle/input/competitions/ieee-fraud-detection/'

train_tx = pd.read_csv(PATH + 'train_transaction.csv')
train_id = pd.read_csv(PATH + 'train_identity.csv')
test_tx  = pd.read_csv(PATH + 'test_transaction.csv')
test_id  = pd.read_csv(PATH + 'test_identity.csv')

train = train_tx.merge(train_id, on='TransactionID', how='left')
test  = test_tx.merge(test_id,  on='TransactionID', how='left')

print("Train shape:", train.shape)
print("Test shape: ", test.shape)
train.head()

Train shape: (590540, 434)
Test shape:  (506691, 433)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [9]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Fraud rate in train: {y_train.mean():.4f}")
print(f"Fraud rate in test:  {y_test.mean():.4f}")

X_train: (472432, 432)
X_test:  (118108, 432)
Fraud rate in train: 0.0350
Fraud rate in test:  0.0350


## Data Cleaning

In [10]:
with mlflow.start_run(run_name="RandomForest_Cleaning"):
    missing = X_train.isnull().mean()
    drop_cols = missing[missing > 0.8].index.tolist()
    X_train = X_train.drop(columns=drop_cols)
    X_test  = X_test.drop(columns=drop_cols)
    print(f"Dropped: {len(drop_cols)} | Remaining: {X_train.shape[1]}")
    mlflow.log_param("missing_threshold", 0.8)
    mlflow.log_param("dropped_cols", len(drop_cols))
    mlflow.log_param("remaining_features", X_train.shape[1])

Dropped: 74 | Remaining: 358
🏃 View run RandomForest_Cleaning at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/9524cd1f38a740bc81c2e698337d0971
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


## Feature Engineering

In [11]:
with mlflow.start_run(run_name="RandomForest_Feature_Engineering"):
    def feature_engineering(df):
        df = df.copy()
        df['hour']  = (df['TransactionDT'] // 3600) % 24
        df['day']   = (df['TransactionDT'] // (3600 * 24)) % 7
        df['month'] = (df['TransactionDT'] // (3600 * 24 * 30)) % 12
        df['log_TransactionAmt'] = np.log1p(df['TransactionAmt'])
        df['cents'] = df['TransactionAmt'] - np.floor(df['TransactionAmt'])
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        df['uid2'] = df['uid'] + '_' + df['card3'].astype(str)
        df['email_match']   = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        df['P_email_count'] = df['P_emaildomain'].map(df['P_emaildomain'].value_counts())
        df['R_email_count'] = df['R_emaildomain'].map(df['R_emaildomain'].value_counts())
        df['card1_count'] = df['card1'].map(df['card1'].value_counts())
        df['uid_count']   = df['uid'].map(df['uid'].value_counts())
        if 'id_30' in df.columns:
            df['OS'] = df['id_30'].str.extract(r'^([a-zA-Z\s]+)')
        if 'id_31' in df.columns:
            df['browser'] = df['id_31'].str.extract(r'^([a-zA-Z\s]+)')
        return df
    X_train = feature_engineering(X_train)
    X_test  = feature_engineering(X_test)
    print(f"Shape after FE: {X_train.shape}")
    mlflow.log_param("total_features", X_train.shape[1])

Shape after FE: (472432, 371)
🏃 View run RandomForest_Feature_Engineering at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/8b93c13b40a44c5b83633cce624ded8e
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


## Encoding and Imputation

In [12]:
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

with mlflow.start_run(run_name="RandomForest_Encoding"):
    cat_cols = X_train.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
        le.fit(combined)
        X_train[col] = le.transform(X_train[col].astype(str))
        X_test[col]  = le.transform(X_test[col].astype(str))
    print(f"Label encoded {len(cat_cols)} columns")
    imputer = SimpleImputer(strategy='median')
    X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
    X_test  = pd.DataFrame(imputer.transform(X_test),      columns=X_test.columns)
    print(f"Imputed. Final shape: {X_train.shape}")
    mlflow.log_param("encoding", "LabelEncoder")
    mlflow.log_param("imputation", "median")
    mlflow.log_param("scaling", "None - not needed for trees")

Label encoded 29 columns
Imputed. Final shape: (472432, 371)
🏃 View run RandomForest_Encoding at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/45c8a4544a7b47a0a124d53f96157967
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


## Feature Selection

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

with mlflow.start_run(run_name="RandomForest_Feature_Selection_v1_mean"):
    correlations = pd.Series(
        np.abs(np.corrcoef(X_train.T, y_train)[-1, :-1]),
        index=X_train.columns
    )
    low_corr_cols = correlations[correlations < 0.005].index.tolist()
    X_train_fs = X_train.drop(columns=low_corr_cols)
    X_test_fs  = X_test.drop(columns=low_corr_cols)
    print(f"Correlation filter removed: {len(low_corr_cols)} features")
    
    # - use shallow RF for feature selection -
    selector_rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
    selector_rf.fit(X_train_fs, y_train)
    selector = SelectFromModel(selector_rf, prefit=True, threshold='mean')
    selected_cols_v1 = X_train_fs.columns[selector.get_support()].tolist()
    print(f"v1 threshold=mean kept: {len(selected_cols_v1)} features")
    mlflow.log_param("selection_method", "rf_importance_mean")
    mlflow.log_metric("final_features", len(selected_cols_v1))

with mlflow.start_run(run_name="RandomForest_Feature_Selection_v2_nonzero"):
    importances = pd.Series(selector_rf.feature_importances_, index=X_train_fs.columns)
    selected_cols_v2 = importances[importances > 0].index.tolist()
    X_train_fs = X_train_fs[selected_cols_v2]
    X_test_fs  = X_test_fs[selected_cols_v2]
    print(f"v2 nonzero kept: {len(selected_cols_v2)} features")
    print(f"Final shape: {X_train_fs.shape}")
    mlflow.log_param("selection_method", "rf_importance_nonzero")
    mlflow.log_metric("final_features", len(selected_cols_v2))

Correlation filter removed: 73 features
v1 threshold=mean kept: 52 features
🏃 View run RandomForest_Feature_Selection_v1_mean at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/b1e0d80d8bc74bd286f82918aed7d950
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
v2 nonzero kept: 249 features
Final shape: (472432, 249)
🏃 View run RandomForest_Feature_Selection_v2_nonzero at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/9199395d8a2a49b3abe379f28b399d85
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


In [14]:
# reset to v1 selection - 52 features
selector_rf_final = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
selector_rf_final.fit(X_train.drop(columns=low_corr_cols), y_train)
selector_final = SelectFromModel(selector_rf_final, prefit=True, threshold='mean')
selected_cols_final = X_train.drop(columns=low_corr_cols).columns[selector_final.get_support()].tolist()

X_train_fs = X_train.drop(columns=low_corr_cols)[selected_cols_final]
X_test_fs  = X_test.drop(columns=low_corr_cols)[selected_cols_final]

print(f"Final features: {X_train_fs.shape[1]}")

Final features: 52


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

# - underfit -
with mlflow.start_run(run_name="RandomForest_Underfit"):
    model = RandomForestClassifier(n_estimators=10, max_depth=3, random_state=42, class_weight='balanced', n_jobs=-1)
    model.fit(X_train_fs, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train_fs)[:,1])
    test_auc  = roc_auc_score(y_test,  model.predict_proba(X_test_fs)[:,1])
    print(f"Underfit | Train: {train_auc:.4f} | Test: {test_auc:.4f}")
    mlflow.log_params({"n_estimators": 10, "max_depth": 3, "class_weight": "balanced"})
    mlflow.log_metrics({"train_auc": train_auc, "test_auc": test_auc})

# - overfit -
with mlflow.start_run(run_name="RandomForest_Overfit"):
    model = RandomForestClassifier(n_estimators=500, max_depth=None, random_state=42, class_weight='balanced', n_jobs=-1)
    model.fit(X_train_fs, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train_fs)[:,1])
    test_auc  = roc_auc_score(y_test,  model.predict_proba(X_test_fs)[:,1])
    print(f"Overfit  | Train: {train_auc:.4f} | Test: {test_auc:.4f}")
    mlflow.log_params({"n_estimators": 500, "max_depth": "None", "class_weight": "balanced"})
    mlflow.log_metrics({"train_auc": train_auc, "test_auc": test_auc})

# - tuned -
for n_est in [25, 50, 100, 150, 250]:
    for depth in [5, 10, 15]:
        with mlflow.start_run(run_name=f"RandomForest_n{n_est}_d{depth}"):
            model = RandomForestClassifier(
                n_estimators=n_est,
                max_depth=depth,
                random_state=42,
                class_weight='balanced',
                n_jobs=-1
            )
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            cv_scores = cross_val_score(model, X_train_fs, y_train, cv=cv, scoring='roc_auc')
            model.fit(X_train_fs, y_train)
            
            y_pred       = model.predict(X_test_fs)
            y_pred_proba = model.predict_proba(X_test_fs)[:,1]
            train_auc    = roc_auc_score(y_train, model.predict_proba(X_train_fs)[:,1])
            test_auc     = roc_auc_score(y_test, y_pred_proba)
            f1           = f1_score(y_test, y_pred)
            precision    = precision_score(y_test, y_pred)
            recall       = recall_score(y_test, y_pred)
            
            print(f"n={n_est} d={depth} | Train: {train_auc:.4f} | Test: {test_auc:.4f} | F1: {f1:.4f} | CV: {cv_scores.mean():.4f}")
            mlflow.log_params({"n_estimators": n_est, "max_depth": depth, "class_weight": "balanced"})
            mlflow.log_metrics({
                "train_auc": train_auc,
                "test_auc": test_auc,
                "cv_auc_mean": cv_scores.mean(),
                "cv_auc_std": cv_scores.std(),
                "test_f1": f1,
                "test_precision": precision,
                "test_recall": recall
            })

Underfit | Train: 0.8122 | Test: 0.8120
🏃 View run RandomForest_Underfit at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/3ef29ed34b5a4d1cbac577473d67a337
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
Overfit  | Train: 0.9750 | Test: 0.8512
🏃 View run RandomForest_Overfit at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/b3c7becf5d5248e89c85b6d237e20a38
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
n=25 d=5 | Train: 0.8357 | Test: 0.8345 | F1: 0.2263 | CV: 0.8332
🏃 View run RandomForest_n25_d5 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/8df302fb13a24eaeb9a085d5611d06f1
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
n=25 d=10 | Train: 0.8659 | Test: 0.8599 | F1: 0.2499 | CV: 0.8596
🏃 View run RandomForest_n25_d10 at: https:/

In [17]:
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report

with mlflow.start_run(run_name="RandomForest_Best_Pipeline"):
    best_model = RandomForestClassifier(
        n_estimators=150,
        max_depth=15,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
    best_model.fit(X_train_fs, y_train)
    
    y_pred       = best_model.predict(X_test_fs)
    y_pred_proba = best_model.predict_proba(X_test_fs)[:,1]
    train_auc    = roc_auc_score(y_train, best_model.predict_proba(X_train_fs)[:,1])
    test_auc     = roc_auc_score(y_test, y_pred_proba)
    f1           = f1_score(y_test, y_pred)
    precision    = precision_score(y_test, y_pred)
    recall       = recall_score(y_test, y_pred)
    
    print(f"AUC: {test_auc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
    print(classification_report(y_test, y_pred))
    
    mlflow.log_params({"n_estimators": 150, "max_depth": 15, "class_weight": "balanced"})
    mlflow.log_metrics({
        "train_auc": train_auc,
        "test_auc": test_auc,
        "test_f1": f1,
        "test_precision": precision,
        "test_recall": recall
    })
    mlflow.sklearn.log_model(
        best_model,
        "random_forest_model",
        registered_model_name="RandomForest_Fraud"
    )
    print("Model saved.")

AUC: 0.8769 | F1: 0.3057 | Precision: 0.1950 | Recall: 0.7070
              precision    recall  f1-score   support

           0       0.99      0.89      0.94    113975
           1       0.20      0.71      0.31      4133

    accuracy                           0.89    118108
   macro avg       0.59      0.80      0.62    118108
weighted avg       0.96      0.89      0.92    118108



2026/05/03 00:45:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 00:45:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'RandomForest_Fraud'.
2026/05/03 00:46:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForest_Fraud, version 1
Created version '1' of model 'RandomForest_Fraud'.


Model saved.
🏃 View run RandomForest_Best_Pipeline at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/9b0ed46b33d64b289602bb66c22019ca
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
